In [27]:
################################# TOPIC 5 - GROUP BY AND AGGREGATION #########################################

In [28]:
# Groupby and aggregation - Summarizing metrics per server, per priority, per time window

In [29]:
# LOAD DATASET TO WORK ON
import pandas as pd
import numpy as np

# ── Load all three datasets ────────────────────────────────────────────
df_metrics = pd.read_csv(
    "server_metrics.csv",
    parse_dates=["timestamp"],
    dtype={"server_id": "category"},
    na_values=["N/A", "null", "—"]
)

df_tickets = pd.read_csv(
    "incidents.csv",
    parse_dates=["created_at", "resolved_at"]
)

df_logs = pd.read_csv(
    "app_logs.csv",
    parse_dates=["timestamp"]
)

# ── Confirm shapes ─────────────────────────────────────────────────────
print(f"df_metrics : {df_metrics.shape}")
print(f"df_tickets : {df_tickets.shape}")
print(f"df_logs    : {df_logs.shape}")

df_metrics : (525, 7)
df_tickets : (213, 7)
df_logs    : (315, 5)


In [30]:
# Basic groupby + single aggregation
df_metrics.groupby("server_id", observed=True)["cpu_pct"].mean()

server_id
srv-01    59.926667
srv-02    59.931429
srv-03    58.565714
srv-04    59.409524
srv-05    62.332381
Name: cpu_pct, dtype: float64

In [31]:
# Multiple aggregations - agg()
# on one column
df_metrics.groupby("server_id", observed=True)["cpu_pct"].agg(["mean", "max", "min", "std"]).round(2)

,mean,max,min,std
server_id,,,,
srv-01,59.93,98.4,21.8,24.37
srv-02,59.93,98.2,20.7,23.06
srv-03,58.57,98.5,20.9,23.10
srv-04,59.41,98.6,20.6,25.00
srv-05,62.33,96.7,20.6,22.01


In [32]:
# Multiple stats on one column
df_metrics.groupby("server_id", observed=True)["cpu_pct"].agg(["mean", "max", "min", "std"]).round(2)

,mean,max,min,std
server_id,,,,
srv-01,59.93,98.4,21.8,24.37
srv-02,59.93,98.2,20.7,23.06
srv-03,58.57,98.5,20.9,23.10
srv-04,59.41,98.6,20.6,25.00
srv-05,62.33,96.7,20.6,22.01


In [33]:
# Naming each aggregation explicitly
df_metrics.groupby("server_id", observed=True).agg(
    avg_cpu = ("cpu_pct", "mean"),
    max_cpu = ("cpu_pct", "max"),
    avg_memory = ("memory_pct", "mean"),
    avg_response = ("response_ms", "mean"),
    total_records = ("cpu_pct", "count")
).round(2)

,avg_cpu,max_cpu,avg_memory,avg_response,total_records
server_id,,,,,
srv-01,59.93,98.4,62.51,643.57,105
srv-02,59.93,98.2,64.46,633.02,105
srv-03,58.57,98.5,65.09,622.77,105
srv-04,59.41,98.6,62.21,632.40,105
srv-05,62.33,96.7,62.10,590.14,105


In [34]:
# Groupby on multiple columns
# Average cpu per server per status
df_metrics.groupby(["server_id", "status"], observed=True)["cpu_pct"].mean().round(2)

server_id  status  
srv-01     critical    58.96
           ok          59.29
           warning     62.57
srv-02     critical    61.67
           ok          57.60
           warning     73.05
srv-03     critical    55.89
           ok          57.33
           warning     68.10
srv-04     critical    62.85
           ok          60.21
           warning     56.25
srv-05     critical    50.13
           ok          61.36
           warning     70.87
Name: cpu_pct, dtype: float64

In [35]:
# Groupby on incidents - ticket analysis
# Count tickets per category
df_tickets.groupby("category")["ticket_id"].count().sort_values(ascending=False)

category
CPU High        50
Disk Full       46
Service Down    46
Memory Leak     36
Network Lag     35
Name: ticket_id, dtype: int64

In [37]:
# Resolution rate per category
df_tickets.groupby("category").agg(
    total = ("ticket_id", "count"),
    resolved = ("resolved", "sum"),
).assign(
    resolution_rate = lambda x:(x["resolved"] / x["total"] * 100 ).round(1)
).sort_values("resolution_rate")

,total,resolved,resolution_rate
category,,,
Network Lag,35,19,54.3
Service Down,46,25,54.3
Disk Full,46,26,56.5
Memory Leak,36,22,61.1
CPU High,50,33,66.0
